# ADS 505 Final Project: Combined EDA and Models 1?4

Review copy assembled from origin/main commit `06d2f76`.

Run from the project folder with `historical_data.csv` available. Required packages: pandas, numpy, matplotlib, seaborn, scikit-learn (with `root_mean_squared_error`), and xgboost. Models 3?4 also use pandas `include_groups=False`.

The original notebooks and their explanations are preserved below in order. Only repeated imports/settings and CSV reloads were removed or replaced. The handoff resets row indices to match the original CSV-loading behavior. Data preparation, model parameters, tuning, metrics, plots, and exports remain unchanged. Running this notebook writes the original partition CSVs and result files. Saved outputs are cleared pending a fresh run.

Model result variable names are reused in the original sections; each section displays and exports its results before the next section starts. The original evaluation and selection approaches remain intact.


## Source notebook: EDA and Pre-Processing.ipynb



# **Predicting Doordash Delivery Duration with Machine Learning**

## ADS-505 Final Team Project
---
**Data source:** Kaggle  
**Team members:** Julia Gonzalez, Caly Nguyen, Aishwarya Kuduvalli

## **Problem Statement**
---
Delivery time is a critical part of the customer experience, especially for a service like DoorDash, where customers expect to know when their food will arrive. A too-short estimate may leave customers frustrated when their order arrives later than expected, while an estimate that is too long may discourage them from placing an order in the first place. Delivery duration can also vary depending on the restaurant, the order details, or how busy the drivers are at the time. By considering these factors together, the team hopes to better understand what affects delivery duration and whether machine learning can help produce more accurate estimates. 

The corresponding objective of this project is to examine the relationship between order, market, and store-related information, and delivery duration, with the aim of estimating the outcome with predictive modeling. With Kaggle’s DoorDash ETA Prediction dataset, the team will perform exploratory data analysis, pre-process the data, develop and compare multiple machine learning models, and select a final model based on performance and potential business impact. The limitations associated with the historical nature of the data, which consists of orders from early 2015, will also be taken into consideration. Results will be presented as an actionable recommendation, determining what contributes to delivery duration and ETA accuracy and how these findings can be used to improve delivery operations and customer experience. 

## **Data Description**
---

The DoorDash ETA prediction dataset consists of 197,428 delivery records from 2015. It describes order details and times, store features, market information, and other correlated estimations, and is sourced from Kaggle. There are a total of 16 variables, and a response variable `delivery_duration` will be created representing the time from when the order was created to actual delivery time. 

## **Variables**
---
- `market_id` - City/region the DoorDash order was made (in ID form, from `1` to `6`)
- `created_at` - UTC timestamp of when the order was made
- `actual_delivery_time` - UTC timestamp of when the order was delivered
- `store_id` - Restaurant the order was made to (in ID form)
- `store_primary_category` - Restaurant's cuisine type (e.g. `american`, `mexican`, `NA`)
- `order_protocol` - Method through which order was received (`1` to `7`)
- `total_items` - Total number of items in the order 
- `subtotal` - Order subtotal (in cents)
- `num_distinct_items` - Number of distinct items in the order
- `min_item_price` - Minimum item price in the order (in cents)
- `max_item_price` - Maximum item price in the order (in cents) 
- `total_onshift_dashers` - Total number of dashers within a 10 mile radius of the store during placement 
- `total_busy_dashers` - Subset of `total_onshift_dashers` who were currently busy with an order
- `total_outstanding_orders` - Number of active orders being processed within a 10 mile radius of the current order
- `estimated_order_place_duration` - Estimated time between order placement and restaurant receiving it (in seconds)
- `estimated_store_to_consumer_driving_duration` - Estimated time to drive from store to customer (in seconds) 

## **Imports**
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

# Sklearn imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

## **Data Loading, Initial Exploration, and Cleaning**
---

### Loading the Data

In [ ]:
doordash = pd.read_csv("historical_data.csv")

# First 5 rows
doordash.head()

### Overview

**Dimensions**

In [ ]:
print("Rows and columns:", doordash.shape)

- 197,428 rows
- 16 columns

**Column Names**

In [ ]:
columns = doordash.columns

print("Column names:")
for c in columns:
    print(c, end = ", ")

**Data Types**

In [ ]:
doordash.dtypes

**Descriptive Statistics**

In [ ]:
doordash.describe().T

### Missing and Duplicate Values

**Duplicated Row Counts**

In [ ]:
duplicated = doordash.duplicated().sum()
print("Duplicated row counts:", duplicated)

**Missing Value Counts**

In [ ]:
nulls = doordash.isnull().sum()
SAMPLESIZE = 197428

# Table of null values
nulltable = pd.DataFrame({"Count": nulls,
                          "Percentage": round(nulls / SAMPLESIZE * 100, 3)})
nulltable

There were missing values identified in `market_id`, `store_primary_category`, `order_protocol`, `total_onshift_dashers`, `total_busy_dashers`, `total_outstanding_orders`, and `estimated_store_to_consumer_driving_duration`. Handling would require either imputation, representing missingness in some form, or deleting rows. 

`total_onshift_dashers`, `total_busy_dashers`, and `total_outstanding_orders` all had the same higher number of missing values of 16,262. Their relationship with each other and the remaining variables was considered further, using contingency tables:

In [ ]:
# Total onshift dashers and total busy dashers
pd.crosstab(doordash["total_onshift_dashers"]
            .isnull(),
            doordash["total_busy_dashers"]
            .isnull())

In [ ]:
# Total onshift dashers and total outstanding orders
round(pd.crosstab(doordash["total_onshift_dashers"]
                  .isnull(),
                  doordash["total_outstanding_orders"]
                  .isnull()), 3)

In [ ]:
# Total busy dashers and total outstanding orders
round(pd.crosstab(doordash["total_busy_dashers"]
                  .isnull(),
             doordash["total_outstanding_orders"]
                  .isnull()), 3)

Neither of these three variables contain missing values independently of one another; missingness in one indicated missingness for the remaining 2. Next, the relationship between the three and `market_id` and `order_protocol` was examined, using using `total_onshift_dashers` to compare. These variables were selected because of their categorical nature and easily identifiable categories.

In [ ]:
# Three-variable missing values
three_missing = doordash["total_onshift_dashers"].isnull()
three_missing.name = "three_missing"

# Missing value relationship with market_id
round(pd.crosstab(doordash["market_id"],
                  three_missing,
                  normalize = "index"), 3)

In [ ]:
# Missing value relationship with order_protocol
round(pd.crosstab(doordash["order_protocol"],
                  three_missing,
                  normalize = "index"), 3)

From these contingency tables, a strong potential relationship between the three variables' missing values and `market_id` became apparent; market `6` had a substantially larger proportion (95.46%) than all other market types. This suggested that missingness among these variables was not random, but correlated with market. 

**Handling Missing Values**

In [ ]:
# Dropping rows with missing actual delivery times
doordash = doordash.dropna(subset = ["actual_delivery_time", 
                                     "total_onshift_dashers",
                                     "total_busy_dashers",
                                     "total_outstanding_orders",
                                     "estimated_store_to_consumer_driving_duration"])

In [ ]:
doordash.isnull().sum()

The missing values in `actual_delivery_time`, `total_onshift_dashers`, `total_busy_dashers`, `total_outstanding_orders`, and `estimated_store_to_consumer_driving_duration`. were addressed by dropping the corresponding rows, the latter three to ensure market conditions were represented using observed instead of imputed values. Imputation to handle the remaining missing values would be performed post-partition to prevent data leakage. Speciifcally, the variables `market_id`, `store_primary_category`, and `order_protocol`, all containing missing values and were slated for mode imputation. 

### Outliers and Impossible Values

**Outliers**

In [ ]:
# Numeric variables
numeric = ["total_items",
           "subtotal",
           "num_distinct_items",
           "min_item_price",
           "max_item_price",
           "total_onshift_dashers",
           "total_busy_dashers",
           "total_outstanding_orders",
           "estimated_order_place_duration",
           "estimated_store_to_consumer_driving_duration"]

In [ ]:
# Checking skew
round(doordash[numeric].skew(), 3)

In [ ]:
# Checking total_items
doordash["total_items"].sort_values(ascending = False).head(10)

In [ ]:
# Anomalous row
doordash.loc[47231]

In [ ]:
# Average price per item
3115 / 411

In [ ]:
# Dropping anomalous row
doordash = doordash.drop(index = 47231)

In [ ]:
# New total_items skew
print(round(doordash["total_items"].skew(), 3))

Using `skew` to check for outliers, `total_items` was found to be highly right skewed at 21.414. Its top 10 values were identified; on its own, a high number of items did not seem unusual, as large orders are possible. However, the maximum value of 411 was significantly higher than the next highest. Further examination of the corresponding row revealed that the order had a subtotal of ＄31.15, making the average price of each item only ＄0.08. Given these inconsistencies, this record was considered potentially invalid and removed, considerably reducing the column's skew. 

**Negative Values**

In [ ]:
# Checking for negativ/e values in each column
print("Negative values:")
for n in numeric:
    negatives = (doordash[n] < 0).sum()
    
    if negatives > 0:
        print(n + ":", negatives)

# Removing rows with negative values
for n in numeric:
    doordash = doordash[doordash[n] >= 0]

In [ ]:
doordash.isnull().sum()

There were negative, and therefore impossible, values found in `min_item_price`, `total_onshift_dashers`, `total_busy_dashers`, and `total_outstanding_orders`. In response, these rows were also dropped.

## **Feature Engineering / Extraction**
---

### Creating the Response Variable

In [ ]:
# Datetime conversion
doordash["created_at"] = pd.to_datetime(doordash["created_at"])
doordash["actual_delivery_time"] = pd.to_datetime(doordash["actual_delivery_time"])

**Response Variable - Delivery Duration**

In [ ]:
# Delivery duration (minutes)
doordash["delivery_duration"] = (doordash["actual_delivery_time"]
                                 - doordash["created_at"]).dt.total_seconds() / 60
doordash["delivery_duration"].describe()

The response variable `delivery_duration` was created by subtracting `created_at` from `actual_delivery_time` (both of which were previously converted to `datetime` format) and calculating the resulting duration in minutes. 

**Delivery Duration Outliers**

In [ ]:
# Number of delivery durations above 5 hours
abovefive = (doordash["delivery_duration"] > 300).sum()

print("Number of delivery durations above 5 hours:", abovefive)

In [ ]:
# Removing rows with delivery duration > 5 hours
doordash = doordash[doordash["delivery_duration"] <= 300]

Outliers in `delivery_duration` were defined as values above 5 hours (300 minutes). 31 records were found to exceed this threshold and were therefore removed.

### Time Variables

**Extracting Time-Related Variables**

In [ ]:
# Hour the order was made
doordash["order_hour"] = doordash["created_at"].dt.hour

# Day of week the order was made
doordash["order_dayofweek"] = doordash["created_at"].dt.dayofweek

# Whether the order was made on a weekend or not
doordash["order_weekend"] = (doordash["created_at"].dt.dayofweek >= 5).astype(int)

**Dropping Original Time Variables**

In [ ]:
doordash = doordash.drop(columns = ["created_at",
                                    "actual_delivery_time"])

The variables `order_hour`, `order_dayofweek`, and `order_weekend` signifying the hour the order was made, day of week the order was made, and whether the order was made on a weekend or not were derived from `created_at`. Next, the original time variables were dropped as their necessary information had been distracted and retaining `actual_delivery_time` would result in data leakage.

### Creating Other Variables

In [ ]:
# Average quantity ordered of each item
doordash["average_quantity_per_item"] = (doordash["total_items"]
                                         / doordash["num_distinct_items"])

The variable `average_quantity_per_item` was also created to potentially provide more information to the analysis regarding the orders. 

### Variable Reorganization

In [ ]:
doordash = doordash[["delivery_duration",
                     "market_id",
                     "store_id",
                     "store_primary_category",
                     "order_hour",
                     "order_dayofweek",
                     "order_weekend",
                     "order_protocol",
                     "total_items",
                     "subtotal",
                     "min_item_price",
                     "max_item_price",
                     "num_distinct_items",
                     "average_quantity_per_item",
                     "total_onshift_dashers",
                     "total_busy_dashers",
                     "total_outstanding_orders",
                     "estimated_order_place_duration",
                     "estimated_store_to_consumer_driving_duration"]]
doordash.head()

## **Exploratory Data Analysis (EDA)**
---

### Distribution of Delivery Duration

In [ ]:
# Histogram of delivery duration
sns.histplot(doordash,
             x = "delivery_duration",
             bins = 40,
             color = "firebrick")

plt.xlabel("Delivery Duration (Minutes)")
plt.ylabel("Count")
plt.title("Delivery Duration Histogram")
plt.show()

A histogram using `sns.histplot` was created to better understand the distribution of the response variable `delivery_duration`, which appeared to be right-skewed. The majority of delivery times fell in the range of about 35 to 60 minutes. Evidently, longer deliveries (above 100 minutes) were observed as well, as reflected by the right side of the distribution extending to approximately 150 minutes. 

### Numeric Variables

**Histograms**

In [ ]:
# Remaining numeric variables
numeric = ["total_items",
           "subtotal",
           "min_item_price",
           "max_item_price",
           "num_distinct_items",
           "average_quantity_per_item",
           "total_onshift_dashers",
           "total_busy_dashers",
           "total_outstanding_orders",
           "estimated_order_place_duration",
           "estimated_store_to_consumer_driving_duration"]

In [ ]:
# Numeric variable histograms
fig, axes = plt.subplots(4, 3,
                         figsize = (12, 12))
axes = axes.flatten()

for i, c in enumerate(numeric):
    sns.histplot(doordash[c],
                 bins = 40,
                 ax = axes[i],
                 color = "firebrick")
    axes[i].set_title(c)

plt.tight_layout()
plt.show()

The distribution of numeric predictors was also examined. Most histograms showed a right-skewed distribution, which was especially true for predictors related directly to the order. Most orders had lower prices and fewer items, which falls in line with the expectation of how individual food delivery orders are generally placed. `estimated_store_to_consumer_driving_duration` demonstrated a bell curve distribution with the peak of orders taking having a driving duration of approximately 500 seconds (8.3 minutes) from store to consumer. `estimated_order_place_duration` also behaved more like a discrete variable; most instances occurred at two specific durations. 

**Bivariate Plots**

In [ ]:
fig, axes = plt.subplots(4, 3, figsize = (12, 12))
axes = axes.flatten()

for i, c in enumerate(numeric):
    sns.scatterplot(data = doordash,
                    x = c,
                    y = "delivery_duration",
                    ax = axes[i],
                    color = "firebrick")
    axes[i].set_title(c)

plt.tight_layout()
plt.show()

`sns.scatterplot` was used to create bivariate plots to better evaluate the relaitonship between the numeric predictors and `delivery_duration`. Most plots showed widespread dispersion, indicating no distinct linear relationship. `estimated_order_place_duration` was concentrated in two specific areas but showed substantial variation in delivery duration. `estimated_store_to_consumer_driving_duration` on the other hand showed a very slight positive trend. 

### Categorical Variables

**Bar Plots**

In [ ]:
# Categorical variables besides store_primary_category and store_id
categorical = ["market_id",
               "order_protocol",
               "order_hour",
               "order_dayofweek",
               "order_weekend"]

fig, axes = plt.subplots(2, 3,
                         figsize = (12, 6))
axes = axes.flatten()

for i, c in enumerate(categorical):
    sns.countplot(data = doordash,
                  x = c,
                  ax = axes[i],
                  color = "firebrick")
    axes[i].set_title(c)

plt.tight_layout()
plt.show()

Bar plots were generated with `sns.countplot` for categorical variables. `market_id` and `order_protocol` were uneven in their distribution. `order_hour` was also varied, with greater counts from 1-4 and 19-23 UTC, which is assumed to reflect regular meal-times and store operating hours. `order_weekend` was also uneven with almost twice as many weekday vs. weekend orders, although this is expected considering the distribution of `order_dayofweek`.

**Boxplots**

In [ ]:
fig, axes = plt.subplots(2, 3,
                         figsize = (12, 6))
axes = axes.flatten()

for i, c in enumerate(categorical):
    sns.boxplot(data = doordash,
                x = c,
                y = "delivery_duration",
                ax = axes[i],
                color = "firebrick",
                showfliers = False)
    axes[i].set_title(c)

plt.tight_layout()
plt.show()

Categorical variables were also analyzed with boxplots. Each boxplot was fairly similar in median, although the highest medians across predictors were as follows:
- `market_id`: 1
- `order_protocol`: 6
- `order_hour`: 14 UTC
- `order_dayofweek`: 0 and 5 (Monday and Saturday). 
- `order_weekend`: 1 (weekend)

### Correlation Heatmap

In [ ]:
correlations = doordash[numeric + ['delivery_duration']].corr()

plt.figure(figsize = (10, 8))
sns.heatmap(correlations,
            annot = True,
            cmap = "coolwarm")

plt.title("Correlation Heatmap")
plt.show()

A correlation heatmap explored the linear relationships between predictors; paler shades or values closer to 0 represented weaker correlations. Correlations with `delivery_duration` were relatively weak; `estimated_store_to_consumer_driving_duration` had the highest (r = 0.24). Between the predictors, strong relationships were found: 

- `total_onshift_dashers` with `total_busy_dashers` (r = 0.94)
- `total_onshift_dashers` with `total_outstanding_orders` (r = 0.94)
- `total_busy_dashers` with `total_outstanding_orders` (r = 0.93)
- `total_items` with `num_distinct_items`: 0.81 (r = 0.81)

However, each of these variables represents distinct information that might be relevant to delivery duration, so they were retained for modeling despite potential multicollinearity. 

## Modeling Preparation
---

### Data Partitioning

**60/20/20 Training-Validation-Test Split**

In [ ]:
training, validationtest = train_test_split(doordash,
                                            test_size = 0.40,
                                            random_state = 1)

validation, test = train_test_split(validationtest,
                                    test_size = 0.50,
                                    random_state = 1)

In [ ]:
# Checking split
print("Training set rows and columns:", training.shape)
print("Validation set rows and columns:", validation.shape)
print("Test set rows and columns:", test.shape)

Note: The data was partitioned using a 60/20/20 train-validation-test split. 

### Missing Value Imputation

**Mode Imputation**

In [ ]:
mode = ["market_id",
        "store_primary_category",
        "order_protocol"]

modeimputer = SimpleImputer(strategy = "most_frequent")

# Imputation
training[mode] = modeimputer.fit_transform(training[mode])
validation[mode] = modeimputer.transform(validation[mode])
test[mode] = modeimputer.transform(test[mode])

In [ ]:
# Checking nulls
print("Training set missing values:", training.isnull().sum().sum())
print("Validation set missing values:", validation.isnull().sum().sum())
print("Test set missing values:", test.isnull().sum().sum())

As mentioned previously, the remaining missing values in `market_id`, `store_primary_category`, and `order_protocol` were handled through mode imputation, using the imputer fitted on the training set subsequently on the validation and test sets to avoid leaked data.

### One-Hot Encoding

**Cardinality**

In [ ]:
# High cardinality variable (training set)
print("Unique store_primary_category values:", training["store_primary_category"].nunique())
print("Unique store_id values:", training["store_id"].nunique())

Both `store_primary_category` and `store_id`, had high cardinality, but the latter significantly more so. Therefore, `store_id` was dropped from the final feature set; otherwise, one-hot encoding would have created thousands of variables and introduced excessive dimensionality. 

In [ ]:
# Removing store_id
training = training.drop(columns = ["store_id"])
validation = validation.drop(columns = ["store_id"])
test = test.drop(columns = ["store_id"])

**Encoding Categorical Variables**

In [ ]:
encode = ["market_id",
          "store_primary_category",
          "order_protocol",
          "order_dayofweek"]

encoder = OneHotEncoder(sparse_output = False,
                        handle_unknown = "ignore")

In [ ]:
# Training set encoded variables
trainingencoded = pd.DataFrame(encoder.fit_transform(training[encode]),
                               columns = encoder.get_feature_names_out(encode),
                               index = training.index)
trainingencoded.head()

In [ ]:
# Validation set encoded variables
validationencoded = pd.DataFrame(encoder.transform(validation[encode]),
                                 columns = encoder.get_feature_names_out(encode),
                                 index = validation.index)
validationencoded.head()

In [ ]:
# Test set encoded variables
testencoded = pd.DataFrame(encoder.transform(test[encode]),
                           columns = encoder.get_feature_names_out(encode),
                           index = test.index)
testencoded.head()

In [ ]:
# Encoded categorical variables in place of original categoricals
training = training.drop(columns = encode).join(trainingencoded)
validation = validation.drop(columns = encode).join(validationencoded)
test = test.drop(columns = encode).join(testencoded)

In [ ]:
# Checking columns
print("Training set rows and columns:", training.shape)
print("Validation set rows and columns:", validation.shape)
print("Test set rows and columns:", test.shape)

The categorical variables `market_id`, `store_primary_category`, `order_protocol`, and `order_dayofweek` were therefore one-hot encoded to ensure their compatibility with the models developed. 

### Exporting

In [ ]:
training.to_csv("training.csv", index = False)
validation.to_csv("validation.csv", index = False)
test.to_csv("test.csv", index = False)

## Source notebook: ADS 505 Final Project - Models 1 and 2.ipynb

Prepared partitions below come directly from the preceding EDA, so a second CSV load is unnecessary. Original standalone CSV instructions are retained for context.


# Models 1 and 2: Linear Regression and XGBoost

## Purpose

The goal of this analysis is to predict DoorDash delivery duration using two different regression approaches. Linear Regression provides a straightforward starting point, while XGBoost allows us to explore whether more complex relationships between the predictors can improve prediction accuracy.

Both models are evaluated using MAE, RMSE, R², and bias, with a median baseline included for comparison. Validation performance is used for model comparison before final evaluation on the test set.

#### Imports and settings

In [ ]:
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from sklearn.inspection import permutation_importance

SEED = 505
TARGET = "delivery_duration"
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)

#### Loading prepared datasets

In [ ]:
# Reuse EDA partitions; match the RangeIndex created by the original CSV loads.
train = training.reset_index(drop=True)
validation = validation.reset_index(drop=True)
test = test.reset_index(drop=True)

print("Training shape:", train.shape)
print("Validation shape:", validation.shape)
print("Test shape:", test.shape)
train.head()

#### Seperate predictors and target

In [ ]:
X_train = train.drop(columns=[TARGET])
y_train = train[TARGET]
X_validation = validation.drop(columns=[TARGET])
y_validation = validation[TARGET]
X_test = test.drop(columns=[TARGET])
y_test = test[TARGET]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

#### Check modeling data



In [ ]:
print("Missing values:")
print("Training:", X_train.isna().sum().sum())
print("Validation:", X_validation.isna().sum().sum())
print("Test:", X_test.isna().sum().sum())
print("\nInfinite values:")
print("Training:", np.isinf(X_train).sum().sum())
print("Validation:", np.isinf(X_validation).sum().sum())
print("Test:", np.isinf(X_test).sum().sum())

In [ ]:
# Compare target distributions across datasets

target_summary = pd.DataFrame({
    "Train": y_train.describe()[["mean", "50%", "std", "min", "max"]],
    "Validation": y_validation.describe()[["mean", "50%", "std", "min", "max"]],
    "Test": y_test.describe()[["mean", "50%", "std", "min", "max"]]
})
target_summary.round(2)

#### Model 1: Linear Regression

Linear Regression provides a simple baseline for predicting delivery duration before comparing it with a more complex model.

In [ ]:
# Fit Linear Regression model

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
print("Linear Regression training complete.")

In [ ]:
# Predict on validation data

linear_val_pred = linear_model.predict(X_validation)
linear_mae = mean_absolute_error(y_validation, linear_val_pred)
linear_rmse = np.sqrt(mean_squared_error(y_validation, linear_val_pred))
linear_r2 = r2_score(y_validation, linear_val_pred)
linear_bias = np.mean(linear_val_pred - y_validation)

print(f"MAE:  {linear_mae:.2f} minutes")
print(f"RMSE: {linear_rmse:.2f} minutes")
print(f"R²:   {linear_r2:.3f}")
print(f"Bias: {linear_bias:.2f} minutes")

In [ ]:
# Compare actual and predicted delivery duration

plt.figure(figsize=(7, 5))
plt.scatter(y_validation, linear_val_pred, alpha=0.2)
min_val = min(y_validation.min(), linear_val_pred.min())
max_val = max(y_validation.max(), linear_val_pred.max())

plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.xlabel("Actual Delivery Duration (minutes)")
plt.ylabel("Predicted Delivery Duration (minutes)")
plt.title("Linear Regression: Actual vs. Predicted")
plt.show()

#### Model 2: XGBoost

XGBoost is used to capture more complex relationships between the predictors and delivery duration.

In [ ]:
# Fit initial XGBoost model
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=SEED,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)
print("XGBoost training complete.")

In [ ]:
# Predict on validation data

xgb_val_pred = xgb_model.predict(X_validation)
xgb_mae = mean_absolute_error(y_validation, xgb_val_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_validation, xgb_val_pred))
xgb_r2 = r2_score(y_validation, xgb_val_pred)
xgb_bias = np.mean(xgb_val_pred - y_validation)

print(f"MAE:  {xgb_mae:.2f} minutes")
print(f"RMSE: {xgb_rmse:.2f} minutes")
print(f"R²:   {xgb_r2:.3f}")
print(f"Bias: {xgb_bias:.2f} minutes")

#### Tuning XGBoost

In [ ]:
# Compare a small set of XGBoost configurations

xgb_configs = [
    {"name": "Current", "n_estimators": 300, "learning_rate": 0.05, "max_depth": 6},
    {"name": "Shallower", "n_estimators": 400, "learning_rate": 0.05, "max_depth": 4},
    {"name": "More Trees", "n_estimators": 500, "learning_rate": 0.03, "max_depth": 6},
    {"name": "Deeper", "n_estimators": 300, "learning_rate": 0.05, "max_depth": 8}
]
xgb_tuning_results = []
for config in xgb_configs:
    model = XGBRegressor(
        n_estimators=config["n_estimators"],
        learning_rate=config["learning_rate"],
        max_depth=config["max_depth"],
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=SEED,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    predictions = model.predict(X_validation)

    xgb_tuning_results.append({
        "Model": config["name"],
        "Trees": config["n_estimators"],
        "Learning Rate": config["learning_rate"],
        "Max Depth": config["max_depth"],
        "MAE": mean_absolute_error(y_validation, predictions)
    })
xgb_tuning_results = pd.DataFrame(xgb_tuning_results)
xgb_tuning_results.sort_values("MAE").round(3)

In [ ]:
# Fit selected XGBoost model

xgb_selected = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=SEED,
    n_jobs=-1
)

xgb_selected.fit(X_train, y_train)
xgb_val_pred = xgb_selected.predict(X_validation)
print("Selected XGBoost model training complete.")

In [ ]:
# Evaluate selected XGBoost model

xgb_mae = mean_absolute_error(y_validation, xgb_val_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_validation, xgb_val_pred))
xgb_r2 = r2_score(y_validation, xgb_val_pred)
xgb_bias = np.mean(xgb_val_pred - y_validation)

print(f"MAE:  {xgb_mae:.2f} minutes")
print(f"RMSE: {xgb_rmse:.2f} minutes")
print(f"R²:   {xgb_r2:.3f}")
print(f"Bias: {xgb_bias:.2f} minutes")

#### Validation Model Comparison

In [ ]:
# Compare validation performance

validation_results = pd.DataFrame({
    "Model": ["Linear Regression", "XGBoost"],
    "MAE": [linear_mae, xgb_mae],
    "RMSE": [linear_rmse, xgb_rmse],
    "R²": [linear_r2, xgb_r2],
    "Bias": [linear_bias, xgb_bias]
})

validation_results.sort_values("MAE").round(3)

In [ ]:
# Compare model prediction errors

plt.figure(figsize=(7, 4))

plt.bar(validation_results["Model"], validation_results["MAE"])

plt.ylabel("Mean Absolute Error (minutes)")
plt.title("Validation MAE by Model")

plt.show()

#### XGBoost: Actual vs. Predicted

In [ ]:
# Compare actual and predicted delivery duration for XGBoost

plt.figure(figsize=(7, 5))
plt.scatter(y_validation, xgb_val_pred, alpha=0.2)

min_val = min(y_validation.min(), xgb_val_pred.min())
max_val = max(y_validation.max(), xgb_val_pred.max())

plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.xlabel("Actual Delivery Duration (minutes)")
plt.ylabel("Predicted Delivery Duration (minutes)")
plt.title("XGBoost: Actual vs. Predicted")
plt.show()

#### Feature Importance by Model

XGBoost identified outstanding orders, dashers on shift, and estimated driving duration as its most important predictors. Order hour also played a larger role compared with Linear Regression, suggesting that delivery times may have more complex relationships with the time an order is placed.

In [ ]:
# Linear Regression permutation importance

linear_importance = permutation_importance(
    linear_model,
    X_validation,
    y_validation,
    scoring="neg_mean_absolute_error",
    n_repeats=3,
    random_state=SEED,
    n_jobs=2
)

linear_features = pd.DataFrame({
    "Feature": X_validation.columns,
    "Importance": linear_importance.importances_mean
}).sort_values("Importance", ascending=False)

linear_features.head(10)

In [ ]:
# XGBoost permutation importance

xgb_importance = permutation_importance(
    xgb_selected,
    X_validation,
    y_validation,
    scoring="neg_mean_absolute_error",
    n_repeats=3,
    random_state=SEED,
    n_jobs=2
)

xgb_features = pd.DataFrame({
    "Feature": X_validation.columns,
    "Importance": xgb_importance.importances_mean
}).sort_values("Importance", ascending=False)

xgb_features.head(10)

In [ ]:
# Compare feature importance across models

top_features = xgb_features.head(8)["Feature"]
importance_comparison = pd.DataFrame({
    "Linear Regression": linear_features.set_index("Feature")["Importance"],
    "XGBoost": xgb_features.set_index("Feature")["Importance"]
}).loc[top_features]

importance_comparison.sort_values("XGBoost").plot(
    kind="barh",
    figsize=(9, 5)
)

plt.xlabel("Increase in Validation MAE (minutes)")
plt.ylabel("Feature")
plt.title("Feature Importance: Linear Regression vs. XGBoost")
plt.legend()
plt.tight_layout()
plt.show()

Both models identified outstanding orders and dashers on shift as the most important predictors. Linear Regression showed greater sensitivity to outstanding orders, while XGBoost relied more on order hour and busy dashers. This suggests that the models use some of the same information differently when predicting delivery duration.

#### Final Model Evaluation

XGBoost was selected based on its lower validation MAE. Both models are now evaluated on the test set to compare their performance on unseen data.

In [ ]:
# Evaluate both models on the test set

linear_test_pred = linear_model.predict(X_test)
xgb_test_pred = xgb_selected.predict(X_test)

test_results = pd.DataFrame({
    "Model": ["Linear Regression", "XGBoost"],
    "MAE": [
        mean_absolute_error(y_test, linear_test_pred),
        mean_absolute_error(y_test, xgb_test_pred)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, linear_test_pred)),
        np.sqrt(mean_squared_error(y_test, xgb_test_pred))
    ],
    "R²": [
        r2_score(y_test, linear_test_pred),
        r2_score(y_test, xgb_test_pred)
    ],
    "Bias": [
        np.mean(linear_test_pred - y_test),
        np.mean(xgb_test_pred - y_test)
    ]
})

test_results.sort_values("MAE").round(3)

XGBoost maintained its advantage on the test set, achieving a lower MAE and RMSE than Linear Regression. Its test performance was also close to its validation results, suggesting the model generalizes reasonably well to similar orders. However, unusually long deliveries remain difficult to predict.

#### Exporting Results

The validation results, test results, and feature importance scores are saved for the team's final model comparison.

In [ ]:
# Export validation and test results

validation_results.to_csv(
    RESULTS / "models_1_2_validation_results.csv",
    index=False
)

test_results.to_csv(
    RESULTS / "models_1_2_test_results.csv",
    index=False
)

# Combine feature importance from both models
importance_results = pd.concat([
    linear_features.assign(Model="Linear Regression"),
    xgb_features.assign(Model="XGBoost")
], ignore_index=True)

importance_results = importance_results[
    ["Model", "Feature", "Importance"]
]

importance_results.to_csv(
    RESULTS / "models_1_2_importance.csv",
    index=False
)
print("Results exported successfully.")

## Source notebook: Caly Models 3 and 4.ipynb

Prepared partitions below come directly from the preceding EDA, so a second CSV load is unnecessary. Original standalone CSV instructions are retained for context.


# Predicting DoorDash Delivery Duration: Models 3 and 4

## ADS 505 Final Team Project

**Caly Nguyen: Random Forest and Histogram Gradient Boosting**

This notebook uses the prepared training, validation, and test files created in
the team's EDA notebook. It tunes both models on training data, selects the
candidate using validation MAE, and reports the test results only after that
selection. The code is intentionally short and uses scikit-learn functions for
loading, tuning, fitting, metrics, and feature importance.

**Required files:** keep `training.csv` (or `train.csv`), `validation.csv`, and
`test.csv` in this folder. The exported files are already imputed and one-hot
encoded by the EDA notebook. Do not fit another imputer or encoder here.

## 1. Imports and settings

The team EDA uses a random 60/20/20 split. These model results therefore
describe a random-holdout evaluation, not a later-in-time forecasting test.
`SEED` makes the model fitting and tuning reproducible.

In [ ]:

from IPython.display import display
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV



## 2. Load the prepared partitions

The helper accepts either common training-file name. It stops with a clear error
if the training partition is missing: a validation or test set cannot be used to
train the models. The checks confirm equal feature columns, no missing values,
and a positive duration target.

In [ ]:
partitions = {"Train": train, "Validation": validation, "Test": test}

for name, frame in partitions.items():
    print(f"{name}: {frame.shape[0]:,} rows, {frame.shape[1]} columns")
    assert frame[TARGET].gt(0).all(), f"{name} has an invalid target"
    assert frame.isna().sum().sum() == 0, f"{name} has missing values"

features = train.columns.drop(TARGET)
assert features.equals(validation.columns.drop(TARGET))
assert features.equals(test.columns.drop(TARGET))


## 3. Create familiar train, validation, and test variables

The target is delivery duration in minutes. All remaining columns are numeric,
preprocessed order-time features from the EDA notebook. We retain the supplied
splits rather than splitting them again.

In [ ]:
X_train, y_train = train[features], train[TARGET]
X_valid, y_valid = validation[features], validation[TARGET]
X_test, y_test = test[features], test[TARGET]

display(pd.DataFrame({
    name: frame[TARGET].describe()[["mean", "50%", "std", "min", "max"]]
    for name, frame in partitions.items()
}).round(2))


## 4. Define and tune Models 3 and 4

Random Forest averages many decision trees. Histogram Gradient Boosting adds
small trees sequentially and uses absolute-error loss, which is less sensitive
to large delivery-duration errors than squared-error loss. `GridSearchCV` uses
three folds from training data only. The small grids keep the process readable
and reproducible instead of searching hundreds of combinations.

In [ ]:
model_specs = {
    "Random Forest": (
        RandomForestRegressor(
            n_estimators=150, max_features=0.8, random_state=SEED, n_jobs=1,
        ),
        {"max_depth": [12, 18], "min_samples_leaf": [10, 25]},
    ),
    "Gradient Boosting": (
        HistGradientBoostingRegressor(
            loss="absolute_error", learning_rate=0.08, max_iter=150,
            early_stopping=False, random_state=SEED,
        ),
        {"max_leaf_nodes": [15, 31], "l2_regularization": [0.0, 1.0]},
    ),
}


def metrics(actual, predicted):
    # Use one definition of the regression metrics for every model.
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": root_mean_squared_error(actual, predicted),
        "R-squared": r2_score(actual, predicted),
        "Bias": (predicted - actual).mean(),
    }


## 5. Fit on training data and select with validation data

The loop replaces nearly identical model-specific code. It tunes each model
inside training, predicts the untouched validation partition, and stores its
best fitted estimator. The median baseline shows whether either model improves
on a simple constant ETA. The model with the lowest validation MAE is selected
before viewing test results.

In [ ]:
fitted_models = {}
validation_rows = []

baseline = DummyRegressor(strategy="median").fit(X_train, y_train)
validation_rows.append({
    "Model": "Median baseline",
    "Parameters": "median training duration",
    **metrics(y_valid, baseline.predict(X_valid)),
})

for name, (model, grid) in model_specs.items():
    search = GridSearchCV(
        model, grid, cv=3, scoring="neg_mean_absolute_error", n_jobs=1,
    ).fit(X_train, y_train)
    fitted_models[name] = search.best_estimator_
    validation_rows.append({
        "Model": name,
        "Parameters": str(search.best_params_),
        **metrics(y_valid, fitted_models[name].predict(X_valid)),
    })

validation_results = pd.DataFrame(validation_rows).set_index("Model")
selected_name = validation_results["MAE"].idxmin()
display(validation_results.round(3))
print("Validation-selected model:", selected_name)


## 6. Explain the validation models

Permutation importance measures how much validation MAE worsens when one feature
is shuffled. It describes predictive usefulness in this model; it does not prove
that changing the feature causes faster or slower deliveries.

In [ ]:
importance_rows = []
importance_sample = validation.sample(min(3000, len(validation)), random_state=SEED)

for name, model in fitted_models.items():
    result = permutation_importance(
        model, importance_sample[features], importance_sample[TARGET],
        scoring="neg_mean_absolute_error", n_repeats=3, random_state=SEED,
        n_jobs=1,
    )
    importance_rows.append(pd.DataFrame({
        "Feature": features,
        "MAE increase": result.importances_mean,
        "Model": name,
    }))

importance = pd.concat(importance_rows, ignore_index=True)
display(importance.groupby("Model").apply(
    lambda frame: frame.nlargest(5, "MAE increase"), include_groups=False,
).round(3))


## 7. Refit selected settings and evaluate the test partition

The selected settings are refit using train plus validation data. Test results
are reported for both models, but the choice remains the validation-selected
model. Do not choose a new winner after seeing the test table.

In [ ]:
X_train_valid = pd.concat([X_train, X_valid])
y_train_valid = pd.concat([y_train, y_valid])
final_models = {}
test_rows = []

for name, model in fitted_models.items():
    final_models[name] = clone(model).fit(X_train_valid, y_train_valid)
    test_rows.append({
        "Model": name,
        **metrics(y_test, final_models[name].predict(X_test)),
    })

final_baseline = DummyRegressor(strategy="median").fit(X_train_valid, y_train_valid)
test_rows.append({
    "Model": "Median baseline",
    **metrics(y_test, final_baseline.predict(X_test)),
})
test_results = pd.DataFrame(test_rows).set_index("Model")
display(test_results.sort_values("MAE").round(3))


## 8. Findings and reproducible exports

The data were randomly split and preprocessed
before export. The results estimate performance on similar historical orders;
they do not establish a causal operational change or confirm performance on a
later independent period.

In [ ]:
selected_test_mae = test_results.loc[selected_name, "MAE"]
baseline_test_mae = test_results.loc["Median baseline", "MAE"]
reduction = 100 * (1 - selected_test_mae / baseline_test_mae)
top_features = importance.loc[
    importance["Model"].eq(selected_name)
].nlargest(3, "MAE increase")["Feature"].tolist()

print(f"Selected model: {selected_name}")
print(f"Validation MAE: {validation_results.loc[selected_name, 'MAE']:.2f} minutes")
print(f"Test MAE: {selected_test_mae:.2f} minutes")
print(f"Median-baseline test MAE: {baseline_test_mae:.2f} minutes")
print(f"MAE reduction versus baseline: {reduction:.1f}%")
print("Top validation features:", ", ".join(top_features))

validation_results.to_csv(RESULTS / "models_3_4_validation_results.csv")
test_results.to_csv(RESULTS / "models_3_4_test_results.csv")
importance.to_csv(RESULTS / "models_3_4_importance.csv", index=False)
